# ASCEND AI — End-to-End AI Career Assistant
## Final CE-2 Machine Learning Notebook

**Purpose:** demonstrate the dataset preparation, NLP, resume-category classification, resume–job similarity, skill extraction, skill-gap analysis, evaluation and model export that support Ascend AI.

**Datasets used:** `resumes(2).csv`, `jobs(2).csv`, `skills(3).csv`, and the engineered `Ascend(2).csv`. Upload all four when prompted. The notebook also accepts the simpler filenames `resumes.csv`, `jobs.csv`, `skills.csv`, and `Ascend.csv`.

> Run cells from top to bottom. All metrics and charts are calculated from the uploaded data at runtime; no scores are hard-coded. Similarity is a text relevance score, not a probability of getting hired.

## 1. Libraries and reproducibility

In [ ]:
import os, re, json, warnings, joblib, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded. Random state:", RANDOM_STATE)

## 2. Upload and load the four project datasets
In Google Colab, upload the four CSV files together when the file picker appears.

In [ ]:
# Colab upload. If running locally, place the CSVs in the current working directory.
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
except ImportError:
    print("Not running in Colab; reading CSVs from the working directory.")

def find_csv(candidates, required=True):
    for name in candidates:
        if os.path.exists(name): return name
    # case-insensitive fallback
    available = {f.lower(): f for f in os.listdir('.') if f.lower().endswith('.csv')}
    for name in candidates:
        if name.lower() in available: return available[name.lower()]
    if required: raise FileNotFoundError(f"Missing one of: {candidates}. Upload the project CSV files.")
    return None

resume_file = find_csv(["resumes(2).csv", "resumes.csv"])
jobs_file = find_csv(["jobs(2).csv", "jobs.csv"])
skills_file = find_csv(["skills(3).csv", "skills.csv"])
engineered_file = find_csv(["Ascend(2).csv", "Ascend.csv"], required=False)

resumes_raw = pd.read_csv(resume_file, encoding="utf-8", encoding_errors="replace")
jobs_raw = pd.read_csv(jobs_file, encoding="utf-8", encoding_errors="replace")
skills_raw = pd.read_csv(skills_file, encoding="utf-8", encoding_errors="replace")
engineered_raw = pd.read_csv(engineered_file, encoding="utf-8", encoding_errors="replace") if engineered_file else None
print("Loaded files:", resume_file, jobs_file, skills_file, engineered_file)

## 3. Dataset overview and schema validation

In [ ]:
print("RESUMES:", resumes_raw.shape, list(resumes_raw.columns))
print("JOBS:", jobs_raw.shape, list(jobs_raw.columns))
print("SKILLS:", skills_raw.shape, list(skills_raw.columns))
if engineered_raw is not None: print("ENGINEERED RESUMES:", engineered_raw.shape, list(engineered_raw.columns))

# Validate expected columns and fail with a useful message rather than silently guessing.
for col in ["Category", "Resume"]:
    if col not in resumes_raw.columns: raise ValueError(f"Resume dataset must contain '{col}'. Found {list(resumes_raw.columns)}")
for col in ["Job Title", "Job Description"]:
    if col not in jobs_raw.columns: raise ValueError(f"Jobs dataset must contain '{col}'. Found {list(jobs_raw.columns)}")
for col in ["label_cleaned", "altLabels"]:
    if col not in skills_raw.columns: raise ValueError(f"Skills dataset must contain '{col}'. Found {list(skills_raw.columns)}")

dataset_summary = pd.DataFrame({
    "Dataset": ["Resumes", "Jobs", "Skills"] + (["Engineered resumes"] if engineered_raw is not None else []),
    "Rows": [len(resumes_raw), len(jobs_raw), len(skills_raw)] + ([len(engineered_raw)] if engineered_raw is not None else []),
    "Columns": [len(resumes_raw.columns), len(jobs_raw.columns), len(skills_raw.columns)] + ([len(engineered_raw.columns)] if engineered_raw is not None else [])
})
display(dataset_summary)
display(resumes_raw.head(3)); display(jobs_raw.head(3)); display(skills_raw.head(3))

## 4. Data quality checks and preprocessing

In [ ]:
resumes = resumes_raw[["Category", "Resume"]].copy()
jobs = jobs_raw[["Job Title", "Job Description"]].copy()
skills = skills_raw[["label_cleaned", "altLabels"]].copy()

# Normalize nulls, whitespace, encoding artefacts and duplicate rows.
def clean_text(value):
    if pd.isna(value): return ""
    text = str(value).replace("\ufeff", " ").replace("\xa0", " ")
    text = text.replace("â€¢", " ").replace("•", " ").replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for col in ["Category", "Resume"]: resumes[col] = resumes[col].map(clean_text)
for col in ["Job Title", "Job Description"]: jobs[col] = jobs[col].map(clean_text)
for col in ["label_cleaned", "altLabels"]: skills[col] = skills[col].map(clean_text)

quality = pd.DataFrame({
    "Dataset": ["Resumes", "Jobs", "Skills"],
    "Duplicate rows before removal": [resumes.duplicated().sum(), jobs.duplicated().sum(), skills.duplicated().sum()],
    "Missing/blank key values": [((resumes.Category == "") | (resumes.Resume == "")).sum(), ((jobs["Job Title"] == "") | (jobs["Job Description"] == "")).sum(), (skills.label_cleaned == "").sum()]
})
display(quality)
resumes = resumes[(resumes.Category != "") & (resumes.Resume != "")].drop_duplicates().reset_index(drop=True)
jobs = jobs[(jobs["Job Title"] != "") & (jobs["Job Description"] != "")].drop_duplicates().reset_index(drop=True)
skills = skills[skills.label_cleaned != ""].drop_duplicates().reset_index(drop=True)
print("After cleaning:", resumes.shape, jobs.shape, skills.shape)

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Category frequency
category_counts = resumes["Category"].value_counts()
plt.figure(figsize=(12, 5)); category_counts.plot(kind="bar")
plt.title("Resume records by professional category"); plt.xlabel("Resume category"); plt.ylabel("Number of resumes")
plt.xticks(rotation=65, ha="right"); plt.tight_layout(); plt.show()

# Resume length and word count are calculated from cleaned text.
resumes["resume_length"] = resumes.Resume.str.len()
resumes["word_count"] = resumes.Resume.str.split().str.len()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(resumes.resume_length, bins=30); ax[0].set_title("Resume character length"); ax[0].set_xlabel("Characters")
ax[1].hist(resumes.word_count, bins=30); ax[1].set_title("Resume word count"); ax[1].set_xlabel("Words")
plt.tight_layout(); plt.show()

# Job-description length and frequent job titles
jobs["description_word_count"] = jobs["Job Description"].str.split().str.len()
plt.figure(figsize=(8, 4)); plt.hist(jobs.description_word_count, bins=30)
plt.title("Job description length distribution"); plt.xlabel("Words per description"); plt.ylabel("Number of jobs"); plt.tight_layout(); plt.show()
plt.figure(figsize=(10, 5)); jobs["Job Title"].value_counts().head(15).sort_values().plot(kind="barh")
plt.title("15 most frequent job titles"); plt.xlabel("Number of listings"); plt.tight_layout(); plt.show()

print("Category count:", resumes.Category.nunique())
print("Resume word-count summary:"); display(resumes.word_count.describe().to_frame())
print("Job description word-count summary:"); display(jobs.description_word_count.describe().to_frame())

## 6. Feature engineering and skill vocabulary preparation

In [ ]:
# Build clean skill vocabulary from ESCO canonical labels while filtering generic English stopwords and non-skill phrases.
IRRELEVANT_TERMS = {
    "less", "plan", "source", "service", "create", "check", "health", "equipment", "operation",
    "maintenance", "energy", "history", "logic", "migration", "scratch", "assembly", "logging",
    "ecosystems", "decommissioning", "improvise", "crafting", "troubleshoot", "consultation",
    "ethics", "aesthetics", "chef", "us", "set-up", "governance", "guidance", "mining", "programming",
    "prayer", "pregnancy", "childbirth", "hair", "obesity", "imitation jewellery", "reflexology",
    "medicines", "natural gas", "petroleum", "politics", "philosophy", "literature", "sociology",
    "criminology", "genetics", "nutrition", "dietetics", "exercise physiology", "food science",
    "fine arts", "pneumatics", "crimping", "sensors", "photography", "solar energy",
    "english", "hindi", "marathi", "tamil", "punjabi", "german", "french", "swedish", "latin",
    "gather data", "unstructured data", "work independently", "work efficiently", "solve problems",
    "solve technical problems", "make decisions", "meet deadlines", "lead a team", "manage resources",
    "cope with stress", "personal development", "company policies", "promote company",
    "represent the company", "resolve conflicts", "working condition", "quality standards",
    "business model", "business processes", "sales activities", "manage inventory", "communication"
}
VERB_PREFIXES = (
    "manage ", "supervise ", "apply ", "control ", "identify ", "perform ", "ensure ", "show ",
    "train ", "lead ", "handle ", "develop ", "maintain ", "operate ", "conduct ", "assist ",
    "advise ", "prepare ", "coordinate ", "monitor ", "check ", "provide ", "use ", "create ",
    "carry out ", "organize ", "organise ", "inspect ", "install ", "repair ", "clean ", "assemble ",
    "collect ", "compile ", "promote ", "represent ", "resolve ", "solve ", "make ", "meet ", "work "
)

canonical_skills = {re.sub(r"_[0-9]+$", "", s).strip() for s in skills.label_cleaned.str.lower().str.strip() if s.strip()}
all_skill_terms = sorted(
    s for s in canonical_skills
    if (len(s) >= 3 or s in {"r", "c"})
    and s not in IRRELEVANT_TERMS
    and "," not in s
    and len(s.split()) <= 3
    and not any(s.startswith(vp) for vp in VERB_PREFIXES)
)
skill_set = set(all_skill_terms)
print("Canonical labels in skills.csv:", len(canonical_skills))
print("Filtered relevant skill terms:", len(all_skill_terms))

# Fast O(N) n-gram word-boundary skill extraction (unigrams, bigrams, trigrams)
def extract_skills(text):
    text = clean_text(text).lower()
    tokens = [t.rstrip('.') for t in re.findall(r"\b[a-z0-9_\+#\.\/]+\b", text) if t.rstrip('.')]
    n = len(tokens)
    found = set()
    for k in (1, 2, 3):
        for i in range(n - k + 1):
            phrase = " ".join(tokens[i:i+k])
            if phrase in skill_set:
                found.add(phrase)
    return sorted(found)

resumes["extracted_skills"] = resumes.Resume.apply(extract_skills)
resumes["skill_count"] = resumes.extracted_skills.str.len()
resumes["resume_length"] = resumes.Resume.str.len()
resumes["word_count"] = resumes.Resume.str.split().str.len()
display(resumes[["Category", "resume_length", "word_count", "skill_count"]].head(10))
print("Mean detected skills per resume:", round(resumes.skill_count.mean(), 2))

plt.figure(figsize=(8, 4)); plt.hist(resumes.skill_count, bins=25)
plt.title("Detected skill count per resume"); plt.xlabel("Distinct skill terms detected"); plt.ylabel("Number of resumes"); plt.tight_layout(); plt.show()

## 7. Feature selection and target encoding

In [ ]:
# Text is the principal predictive input; category is the supervised target.
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(resumes.Category)
X_text = resumes.Resume.astype(str)
print("Target classes:", len(label_encoder.classes_))
print("Class names:", list(label_encoder.classes_))
print("Text samples:", len(X_text))

# Numeric engineered features are retained for analysis and the resume profile,
# while TF-IDF is used for the NLP classification experiment.
engineered_features = resumes[["resume_length", "word_count", "skill_count"]].copy()
print("Selected engineered features:", list(engineered_features.columns))
display(engineered_features.describe())

## 8. Train-test split and TF-IDF vectorization

In [ ]:
# Stratified split preserves category proportions in train and test sets.
class_counts = pd.Series(y).value_counts()
if class_counts.min() < 2:
    raise ValueError("At least two examples per category are required for a stratified train/test split. Check dataset duplicates/classes.")
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

# TF-IDF: term frequency × inverse document frequency, using unigrams and bigrams.
# Vectorizer is inside each Pipeline so it is fitted only on training data (prevents leakage).
print("TF-IDF configuration: max_features=5000, ngram_range=(1,2), stop_words='english'")

## 9. Model 1 — Logistic Regression

In [ ]:
logistic_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2), sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE))
])
logistic_model.fit(X_train, y_train)
pred_lr = logistic_model.predict(X_test)
print("Logistic Regression training complete.")
print("Example predicted category:", label_encoder.inverse_transform(pred_lr[:5]))

## 10. Model 2 — Linear Support Vector Machine

In [ ]:
svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2), sublinear_tf=True)),
    ("classifier", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
])
svm_model.fit(X_train, y_train)
pred_svm = svm_model.predict(X_test)
print("Linear SVM training complete.")
print("Example predicted category:", label_encoder.inverse_transform(pred_svm[:5]))

## 11. Model evaluation — accuracy, precision, recall and F1-score

In [ ]:
def metric_row(name, y_true, y_pred):
    return {"Model": name, "Accuracy": accuracy_score(y_true, y_pred),
            "Precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
            "Recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
            "F1-score (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0)}

metrics_df = pd.DataFrame([metric_row("Logistic Regression", y_test, pred_lr), metric_row("Linear SVM", y_test, pred_svm)])
display(metrics_df.style.format({c: "{:.4f}" for c in metrics_df.columns if c != "Model"}))
best_name = metrics_df.sort_values("F1-score (weighted)", ascending=False).iloc[0]["Model"]
best_model = logistic_model if best_name == "Logistic Regression" else svm_model
print("Best model by weighted F1-score:", best_name)

## 12. Classification report and confusion matrices

In [ ]:
print("LOGISTIC REGRESSION — classification report")
print(classification_report(y_test, pred_lr, labels=np.arange(len(label_encoder.classes_)),
                            target_names=label_encoder.classes_, zero_division=0))
print("LINEAR SVM — classification report")
print(classification_report(y_test, pred_svm, labels=np.arange(len(label_encoder.classes_)),
                            target_names=label_encoder.classes_, zero_division=0))

# Display confusion matrices. For many categories, labels are rotated and figure size scales.
fig, axes = plt.subplots(1, 2, figsize=(max(14, len(label_encoder.classes_)*0.55), 6))
for ax, pred, title in [(axes[0], pred_lr, "Logistic Regression"), (axes[1], pred_svm, "Linear SVM")]:
    cm = confusion_matrix(y_test, pred, labels=np.arange(len(label_encoder.classes_)))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(title + " — Confusion Matrix"); ax.set_xlabel("Predicted category"); ax.set_ylabel("Actual category")
    ax.set_xticks(range(len(label_encoder.classes_))); ax.set_yticks(range(len(label_encoder.classes_)))
    ax.set_xticklabels(label_encoder.classes_, rotation=90, fontsize=7); ax.set_yticklabels(label_encoder.classes_, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 13. Visual comparison of model performance

In [ ]:
metric_cols = ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"]
ax = metrics_df.set_index("Model")[metric_cols].T.plot(kind="bar", figsize=(10, 5))
ax.set_title("Logistic Regression vs Linear SVM"); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05); ax.set_xlabel("Evaluation metric")
plt.xticks(rotation=15, ha="right"); plt.grid(axis="y", alpha=.25); plt.tight_layout(); plt.show()

## 14. Stratified cross-validation

In [ ]:
# Cross-validation gives a second estimate of generalization.
# Use up to 5 folds, limited by the smallest class count.
min_class_count = int(pd.Series(y).value_counts().min())
n_splits = min(5, min_class_count)
if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    cv_lr = cross_val_score(logistic_model, X_text, y, cv=cv, scoring="f1_weighted", n_jobs=-1)
    cv_svm = cross_val_score(svm_model, X_text, y, cv=cv, scoring="f1_weighted", n_jobs=-1)
    cv_df = pd.DataFrame({"Fold": np.arange(1, n_splits+1), "Logistic Regression": cv_lr, "Linear SVM": cv_svm})
    display(cv_df); print("LR mean ± std:", round(cv_lr.mean(),4), "±", round(cv_lr.std(),4))
    print("SVM mean ± std:", round(cv_svm.mean(),4), "±", round(cv_svm.std(),4))
    cv_df.set_index("Fold").plot(marker="o", figsize=(8,4), ylim=(0,1.05), title="Stratified CV weighted F1 by fold")
    plt.ylabel("Weighted F1-score"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
else:
    print("Cross-validation skipped: a category has fewer than 2 examples.")

## 15. Resume–job matching using TF-IDF and cosine similarity

In [ ]:
def resume_job_similarity(resume_text, job_description):
    docs = [clean_text(resume_text), clean_text(job_description)]
    if not docs[0] or not docs[1]: return 0.0
    vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    matrix = vectorizer.fit_transform(docs)
    return float(cosine_similarity(matrix[0:1], matrix[1:2])[0,0] * 100)

# Demonstrate on a sample of available real records. This is a similarity score, not a hiring probability.
match_rows = []
for i in range(min(10, len(resumes))):
    r = resumes.iloc[i]
    # Compare against a reproducible sample of job descriptions; do not assume category names map to job titles.
    j = jobs.iloc[i % len(jobs)]
    score = resume_job_similarity(r.Resume, j["Job Description"])
    match_rows.append({"Resume category": r.Category, "Job title": j["Job Title"], "Similarity (%)": round(score, 2)})
matching_demo = pd.DataFrame(match_rows)
display(matching_demo)
plt.figure(figsize=(10,4)); plt.bar(range(len(matching_demo)), matching_demo["Similarity (%)"])
plt.xticks(range(len(matching_demo)), [f"Pair {i+1}" for i in range(len(matching_demo))]); plt.ylabel("Cosine similarity (%)")
plt.title("Resume–job text similarity on sample pairs"); plt.tight_layout(); plt.show()

## 16. Skill extraction and skill-gap analysis

In [ ]:
def skill_gap(resume_text, job_description):
    resume_found = set(extract_skills(resume_text))
    job_found = set(extract_skills(job_description))
    matched = sorted(resume_found & job_found)
    missing = sorted(job_found - resume_found)
    coverage = (len(matched) / len(job_found) * 100) if job_found else np.nan
    return matched, missing, coverage, sorted(job_found)

# Analyze first 10 resume/job pairs and show matched and missing terms.
gap_rows = []
for i in range(min(10, len(resumes), len(jobs))):
    r, j = resumes.iloc[i], jobs.iloc[i]
    matched, missing, coverage, required = skill_gap(r.Resume, j["Job Description"])
    gap_rows.append({"Resume category": r.Category, "Job title": j["Job Title"],
                     "Required skills found": len(required), "Matched skills": len(matched),
                     "Missing skills": len(missing), "Skill coverage (%)": coverage})
    print(f"\nPAIR {i+1}: {r.Category} resume vs {j['Job Title']}")
    print("Matched:", matched[:30]); print("Missing:", missing[:30])
gap_df = pd.DataFrame(gap_rows)
display(gap_df)
valid_gap = gap_df.dropna(subset=["Skill coverage (%)"])
if not valid_gap.empty:
    plt.figure(figsize=(10,4)); plt.bar(range(len(valid_gap)), valid_gap["Skill coverage (%)"])
    plt.xticks(range(len(valid_gap)), [f"Pair {i+1}" for i in valid_gap.index], rotation=0)
    plt.ylabel("Coverage (%)"); plt.ylim(0,100); plt.title("Skill coverage across sample resume–job pairs")
    plt.tight_layout(); plt.show()
else:
    print("No identifiable skills were found in the selected job-description samples; no coverage graph can be calculated.")

## 17. Skill vocabulary and dataset visualizations

In [ ]:
# Most frequent skill terms detected in the resume corpus.
from collections import Counter
skill_counts = Counter(skill for skill_list in resumes.extracted_skills for skill in skill_list)
top_skills = pd.DataFrame(skill_counts.most_common(15), columns=["Skill", "Resume frequency"])
display(top_skills)
if not top_skills.empty:
    plt.figure(figsize=(9,5)); plt.barh(top_skills.Skill[::-1], top_skills["Resume frequency"][::-1])
    plt.title("Top 15 extracted skills across resumes"); plt.xlabel("Number of resumes containing skill"); plt.tight_layout(); plt.show()

# Compare engineered dataset values if available; use it as an independent prepared dataset check.
if engineered_raw is not None:
    print("Engineered dataset columns:", list(engineered_raw.columns))
    display(engineered_raw.head())
    numeric_cols = [c for c in ["resume_length", "word_count", "skill_count"] if c in engineered_raw.columns]
    if numeric_cols:
        display(engineered_raw[numeric_cols].describe())
        engineered_raw[numeric_cols].hist(figsize=(10,3), bins=20); plt.suptitle("Precomputed feature distributions in Ascend dataset"); plt.tight_layout(); plt.show()

## 18. Results and interpretation

In [ ]:
print("FINAL EXPERIMENT SUMMARY")
print("------------------------")
print("Resume records used:", len(resumes))
print("Job descriptions used:", len(jobs))
print("Skill vocabulary terms:", len(all_skill_terms))
print("Resume categories:", len(label_encoder.classes_))
print("Selected model by weighted F1:", best_name)
display(metrics_df)

print("Interpretation guide:")
print("- Accuracy: fraction of category predictions that are correct.")
print("- Precision: weighted proportion of predicted category labels that are correct.")
print("- Recall: weighted proportion of actual category records recovered.")
print("- F1: weighted harmonic mean of precision and recall.")
print("- Cosine similarity: overlap in TF-IDF text direction; it is not an ATS standard or hiring probability.")
print("- Skill coverage: matched detected job skills divided by detected job skills; vocabulary coverage is limited by the skill dictionary.")

## 19. Limitations and future scope

### Limitations
- Resume category labels are the dataset targets; the classifier predicts a resume's category, not candidate quality or hiring success.
- The job dataset is not explicitly paired with each resume, so the sample matching demonstration compares reproducible rows and should not be interpreted as ground-truth relevance.
- TF-IDF and cosine similarity are lexical baselines; they may miss semantic equivalence and context.
- Skill extraction depends on the supplied vocabulary and alternative labels; skills absent from the dictionary can be missed.
- A similarity or coverage score should not be presented as an official ATS score.

### Future scope
- Create a manually reviewed resume–job relevance dataset and evaluate ranking with Precision@K, Recall@K and NDCG.
- Add sentence-transformer embeddings for semantic matching and compare against TF-IDF.
- Improve PDF/DOCX parsing, skill normalization and synonym handling.
- Add course recommendation evaluation using verified course metadata and learner feedback.
- Evaluate fairness and category-wise error rates before real recruitment use.

## 20. Export trained models and supporting artifacts

In [ ]:
os.makedirs("ascend_ai_artifacts", exist_ok=True)
joblib.dump(logistic_model, "ascend_ai_artifacts/logistic_regression_tfidf.joblib")
joblib.dump(svm_model, "ascend_ai_artifacts/linear_svm_tfidf.joblib")
joblib.dump(label_encoder, "ascend_ai_artifacts/label_encoder.joblib")
joblib.dump(all_skill_terms, "ascend_ai_artifacts/skill_vocabulary.joblib")

artifact_metrics = {
    "selected_model": best_name,
    "records": int(len(resumes)),
    "categories": int(len(label_encoder.classes_)),
    "test_records": int(len(y_test)),
    "metrics": metrics_df.to_dict(orient="records"),
    "similarity_method": "TF-IDF + cosine similarity",
    "skill_method": "ESCO label and alternative-label dictionary matching"
}
with open("ascend_ai_artifacts/evaluation_metrics.json", "w") as f: json.dump(artifact_metrics, f, indent=2)

# Reload and verify the exported classifier produces identical predictions.
reloaded = joblib.load("ascend_ai_artifacts/linear_svm_tfidf.joblib")
assert np.array_equal(reloaded.predict(X_test), svm_model.predict(X_test))
print("Artifacts exported and reload verification passed.")
print(os.listdir("ascend_ai_artifacts"))

## 21. Download the artifacts (Google Colab)

In [ ]:
# Creates a ZIP containing model files and evaluation metrics.
import shutil
shutil.make_archive("ascend_ai_artifacts", "zip", "ascend_ai_artifacts")
try:
    from google.colab import files
    files.download("ascend_ai_artifacts.zip")
except ImportError:
    print("ZIP saved in the current working directory: ascend_ai_artifacts.zip")

## Final conclusion
Ascend AI combines resume text classification, TF-IDF representation, cosine-similarity matching and dictionary-based skill-gap analysis. The notebook provides a reproducible academic workflow: inspect and clean the supplied datasets, engineer resume features, train and evaluate two supervised classifiers, demonstrate job-text matching, inspect missing skills, visualize the results and export reusable model artifacts. Use the actual values printed by your run in the CE-2 presentation and report.